# 에이전트 계획 수립(Agent Planning)

planning은 agent가 단순히 "답을 쓴다"가 아니라, 문제를 어떤 순서로 풀지 먼저 구조화하는 단계다. 이 노트북은 ReAct, planner-executor, decomposition 세 전략을 비교하면서, 어떤 질문에 어떤 planning 방식이 더 잘 맞는지 살펴본다.

## 학습 목표
- ReAct, planner-executor, decompose 전략의 차이를 설명할 수 있다.
- `PlanGenerator`가 같은 질문을 전략별로 어떻게 다른 step list로 바꾸는지 읽을 수 있다.
- `PlanExecutor`가 handler를 등록하고 step별로 실행하는 구조를 이해한다.
- 계획 길이(plan length)와 질문 복잡도의 관계를 해석할 수 있다.


## 개념 설명

planning 전략은 agent의 reasoning 스타일을 결정한다. ReAct는 관찰-생각-행동을 짧게 반복하며, planner-executor는 먼저 계획을 적고 그 계획을 따라 실행하며, decompose는 넓은 질문을 더 작은 하위 문제로 쪼갠다. 세 전략은 겹치는 부분도 있지만, 어떤 질문에서 더 효율적인지가 다르다.

**목적**
- planning이 왜 retrieval이나 synthesis와 별도 계층으로 필요한지 이해한다.

**핵심 로직**
- 질문을 곧바로 답하지 않고 절차로 바꾸면 복합 질문에서 누락을 줄일 수 있다.
- 전략에 따라 step granularity와 실행 방식이 달라진다.

**결과 해석 가이드**
- 이 notebook에서는 "어느 전략이 최고인가"보다 "어느 상황에 어떤 전략이 맞는가"를 읽는 것이 중요하다.

**💡 면접 포인트**
- "Planner는 모델을 더 똑똑하게 만든다기보다, reasoning을 더 구조적으로 만든다"고 말하면 좋다.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(sys.executable)

## 구현 준비

아래 셀은 planning 실험에 필요한 `PlanGenerator`와 `PlanExecutor`를 준비한다. 먼저 `PlanStep` 자료구조를 보면, 각 step이 단순 문장이 아니라 `step_id`, `objective`, `rationale`, `tool_hint`를 가진다는 점이 보인다. 즉 계획은 설명 가능한 실행 단위다.

**목적**
- plan step이 어떤 정보 단위로 표현되는지 본다.

**핵심 로직**
- `PlanStep`은 실행 전 계획의 최소 단위다.
- `tool_hint`가 있으면 executor가 적절한 handler를 선택할 수 있다.

**실제 소스 코드: PlanStep — src/planner_extended.py**
```python
@dataclass(slots=True)
class PlanStep:
    step_id: str
    objective: str
    rationale: str
    tool_hint: str | None = None

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)
```

**코드 읽기 포인트**
- `step_id`는 trace나 execution log에서 특정 계획 단계를 가리키는 안정적 키다.
- `rationale`이 별도로 있어서 "왜 이 단계를 넣었는가"까지 설명 가능하다.
- `tool_hint`는 planner와 executor의 연결점이다.

**결과 해석 가이드**
- 이 자료구조가 명확할수록 planning을 단순 자연어 설명이 아니라 실행 가능한 계약으로 볼 수 있다.


In [ ]:
import pandas as pd


from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

from src.planner_extended import PlanExecutor, PlanGenerator

pd.set_option('display.max_colwidth', 140)
generator = PlanGenerator()


## 계획 전략 비교

이제 같은 작업을 세 가지 전략으로 계획해 본다. 비교의 핵심은 step 수보다 step의 성격이다. ReAct는 관찰과 판단을 교차시키고, planner-executor는 계획과 실행을 분리하며, decompose는 하위 문제 목록을 강조한다.

**목적**
- 같은 과제가 전략에 따라 어떻게 다른 plan으로 펼쳐지는지 본다.

**핵심 로직**
- `generate_plan()`은 strategy 인자를 받아 분기한다.
- `compare_strategies()`는 세 전략 결과를 한 번에 묶는다.

**주요 파라미터/변수**
- `strategy`: react / planner_executor / decompose
- `planning_task`: 전략별 차이를 볼 공통 작업

**실제 소스 코드: PlanGenerator.generate_plan() — src/planner_extended.py**
```python
    def generate_plan(self, task: str, strategy: str = "planner_executor") -> list[dict[str, Any]]:
        normalized_task = normalize_text(task)
        strategy_name = strategy if strategy in self.STRATEGIES else "planner_executor"

        if strategy_name == "react":
            steps = [
                PlanStep("react_1", f"Observe the task: {normalized_task}", "Start by reading the request carefully."),
                PlanStep("react_2", "Think about what information is missing.", "Identify gaps before acting."),
                PlanStep("react_3", "Act with the best available tool or retrieval step.", "Collect evidence or perform a calculation.", "tool_or_retrieval"),
                PlanStep("react_4", "Observe the result and check whether it is enough.", "Use tool output to decide the next move."),
                PlanStep("react_5", "Respond with the grounded result.", "Finish only after the evidence supports the answer."),
            ]
            return [step.to_dict() for step in steps]

        if strategy_name == "decompose":
            clauses = self._decompose_task(normalized_task)
            steps = [
                PlanStep(
                    step_id=f"decompose_{index}",
                    objective=clause,
                    rationale="Solve one sub-problem at a time to reduce cognitive load.",
                    tool_hint="retrieval" if "find" in clause.lower() or "lookup" in clause.lower() else None,
                )
                for index, clause in enumerate(clauses, start=1)
            ]
            return [step.to_dict() for step in steps]

        steps = [
            PlanStep("plan_1", f"Clarify the objective: {normalized_task}", "Define what a successful answer should contain."),
            PlanStep("plan_2", "Gather evidence for each important part of the task.", "Planner-executor systems work best with explicit evidence collection.", "retrieval"),
            PlanStep("plan_3", "Execute any focused tools needed for calculations or lookups.", "Use tools only after the sub-tasks are clear.", "tool"),
            PlanStep("plan_4", "Synthesize the final response from evidence and tool outputs.", "Combine the executor outputs into a coherent answer."),
        ]
        return [step.to_dict() for step in steps]
```

**코드 읽기 포인트**
- ReAct branch는 5단계 observe-think-act-observe-respond 루프를 고정 템플릿으로 만든다.
- planner_executor branch는 목표 명확화 → 증거 수집 → 도구 실행 → 합성 순으로 간다.
- decompose branch는 `_decompose_task()` 결과를 step으로 바꾼다.
- `compare_strategies()`는 같은 task를 세 전략에 동시에 태워 비교 실험을 쉽게 만든다.

**결과 해석 가이드**
- step 수가 길다고 무조건 좋은 것이 아니다. 질문에 비해 과한 planning은 latency만 늘릴 수 있다.
- 복합 질문일수록 decompose나 planner-executor가 자연스럽고, 짧은 상호작용형 문제는 ReAct가 직관적일 수 있다.


In [ ]:
planning_task = 'Find the rollout date, compare it to the pilot window, and summarize why the timing matters.'
strategy_comparison = generator.compare_strategies(planning_task)
{
    strategy: len(plan)
    for strategy, plan in strategy_comparison.items()
}


## 계획 구조 읽기

아래 표는 전략별 plan을 실제 step 수준에서 읽게 해 준다. 특히 ReAct, planner-executor, decompose가 각각 어떤 사고 습관을 코드에 박아 넣는지 비교해서 보는 것이 중요하다.

**목적**
- 전략별 step objective와 rationale 차이를 눈으로 비교한다.

**핵심 로직**
- ReAct는 상황 관찰과 재평가가 반복된다.
- planner-executor는 먼저 구조를 정한 뒤 실행한다.
- decompose는 큰 질문을 하위 task 리스트로 쪼갠다.

**실제 소스 코드: PlanGenerator._decompose_task() — src/planner_extended.py**
```python
    def _decompose_task(self, task: str) -> list[str]:
        separators = (" and ", " then ", ",")
        parts = [task]
        for separator in separators:
            if separator in task.lower():
                parts = [normalize_text(part) for part in task.split(separator) if normalize_text(part)]
                break
        if len(parts) == 1:
            return [
                f"Identify the core request in: {task}",
                "Collect the evidence needed for the request.",
                "Produce a grounded final answer.",
            ]
        return [f"Handle sub-task: {part}" for part in parts]
```

**코드 읽기 포인트**
- `separators = (" and ", " then ", ",")`는 자연어 task를 대략적인 하위 절로 자르는 기준이다.
- 분해가 불가능하면 기본 3단계 템플릿으로 fallback한다.
- 즉 decompose 전략은 완전한 parser가 아니라, inspectable heuristic planner다.

**결과 해석 가이드**
- ReAct plan은 reasoning loop 성격이 강하고, decompose plan은 task list 성격이 강하다.
- planner_executor는 execution-ready plan이라는 느낌으로 읽으면 된다.


In [ ]:
plan_frames = {
    strategy: pd.DataFrame(plan)
    for strategy, plan in strategy_comparison.items()
}
plan_frames['react'], plan_frames['planner_executor'], plan_frames['decompose']


## PlanExecutor 이해하기

planner가 계획만 잘 세워도 실행기가 없으면 실제 agent가 되지 않는다. `PlanExecutor`는 각 step의 `tool_hint`를 보고 적절한 handler를 찾아 호출하는 아주 작은 실행기다. 이 패턴의 장점은 planning과 execution을 느슨하게 연결해, 새 도구를 붙여도 planner를 크게 바꾸지 않아도 된다는 점이다.

**목적**
- 계획 단계와 실행 단계를 분리하는 executor 패턴을 이해한다.

**핵심 로직**
- `register_handler()`로 hint별 실행기를 등록한다.
- `execute()`는 각 step마다 tool_hint에 맞는 handler를 선택한다.
- 등록된 handler가 없으면 `_default_handler()`가 동작한다.

**주요 파라미터/변수**
- `tool_hint`: 어떤 handler를 고를지 결정하는 planner의 힌트
- `execution_log`: step별 실행 결과 기록

**실제 소스 코드: PlanExecutor — src/planner_extended.py**
```python
class PlanExecutor:
    def __init__(self) -> None:
        self.handlers: dict[str, Handler] = {}

    def register_handler(self, name: str, handler: Handler) -> None:
        self.handlers[name] = handler

    def execute(self, plan: list[dict[str, Any]]) -> list[dict[str, Any]]:
        execution_log: list[dict[str, Any]] = []
        for step in plan:
            tool_hint = step.get("tool_hint")
            handler = self.handlers.get(tool_hint or "", self._default_handler)
            result = handler(step)
            execution_log.append(
                {
                    "step_id": step["step_id"],
                    "objective": step["objective"],
                    "tool_hint": tool_hint,
                    "status": result.get("status", "completed"),
                    "output": result.get("output", ""),
                }
            )
        return execution_log

    @staticmethod
    def _default_handler(step: dict[str, Any]) -> dict[str, Any]:
        return {
            "status": "completed",
            "output": f"Executed step: {step['objective']}",
        }
```

**코드 읽기 포인트**
- `handler = self.handlers.get(tool_hint or "", self._default_handler)`: 등록된 handler가 없으면 기본 실행기로 떨어진다.
- executor는 step 구조를 바꾸지 않고 output만 추가해 주므로, planner 결과와 execution log를 쉽게 비교할 수 있다.
- 즉 planner-executor 아키텍처의 핵심은 "먼저 계획, 나중 실행, 둘 사이를 hint로 연결"하는 것이다.

**결과 해석 가이드**
- execution log에서 `tool_hint`와 `status`를 같이 보면, planner 의도와 executor 동작이 맞았는지 읽을 수 있다.

**💡 면접 포인트**
- "PlanExecutor는 handler registry를 통해 확장 가능성을 확보한다"고 말하면 구조적 이해를 보여줄 수 있다.


In [ ]:
executor = PlanExecutor()
executor.register_handler('retrieval', lambda step: {'status': 'completed', 'output': f"Retrieved evidence for: {step['objective']}"})
executor.register_handler('tool', lambda step: {'status': 'completed', 'output': f"Ran a focused tool for: {step['objective']}"})
planner_executor_plan = generator.generate_plan(planning_task, strategy='planner_executor')
execution_log = executor.execute(planner_executor_plan)
pd.DataFrame(execution_log)


## 실험

여러 과제에 decompose 전략을 적용해 보면, 질문 구조에 따라 plan length와 first step이 달라지는 모습을 볼 수 있다. 이 실험은 planner가 정말로 task decomposition을 하고 있는지, 아니면 매번 같은 템플릿을 기계적으로 내는지 확인하는 데 유용하다.

**목적**
- planning 전략이 입력 작업에 따라 어떻게 달라지는지 본다.

**결과 해석 가이드**
- `plan_length`가 늘어났다면 task를 더 잘게 쪼갠 것이다.
- `first_step`를 보면 planner가 그 질문의 핵심을 어떻게 해석했는지 감이 온다.


In [ ]:
experiment_tasks = [
    'Summarize the rollout plan.',
    'Find the rollout date and explain who needs to know it.',
    'Search the policy goals, calculate the pilot duration, and draft a short update.',
]
experiment_rows = []
for task in experiment_tasks:
    plan = generator.generate_plan(task, strategy='decompose')
    experiment_rows.append({'task': task, 'plan_length': len(plan), 'first_step': plan[0]['objective']})
pd.DataFrame(experiment_rows)


## 결과 해석

마지막 표는 전략별 강점을 한 줄로 요약한다. 읽을 때는 "어떤 전략이 내 시스템 기본값이어야 하는가"보다 "어떤 질문군(question family)에 어떤 전략이 어울리는가"를 중심으로 생각하면 좋다.

**목적**
- 전략 선택 기준을 정리한다.

**결과 해석 가이드**
- ReAct는 인터랙티브한 탐색형 작업, planner-executor는 안정적이고 설명 가능한 실행, decompose는 넓은 작업 분해에 강하다.


In [ ]:
analysis_frame = pd.DataFrame(
    [
        {'strategy': 'react', 'strength': 'good for iterative observe-think-act loops'},
        {'strategy': 'planner_executor', 'strength': 'good for explicit planning and safe execution'},
        {'strategy': 'decompose', 'strength': 'good for turning broad tasks into manageable chunks'},
    ]
)
analysis_frame


## 핵심 정리

이 노트북을 통해 planning은 답변 이전에 사고 과정을 구조화하는 계층이라는 점을 확인했다. ReAct는 짧은 observe-think-act 루프에 강하고, planner-executor는 계획과 실행의 분리가 명확하며, decompose는 넓은 문제를 하위 과제로 나누는 데 유리하다. 어떤 전략이 더 좋은지는 절대값이 아니라 질문 종류와 제품 목표에 따라 달라진다.

또한 `PlanExecutor`의 handler registry 구조를 보면, planning과 execution을 느슨하게 결합하는 패턴이 왜 실용적인지도 드러난다. 새 도구나 실행기를 추가해도 planner 전체를 다시 설계하지 않아도 되기 때문이다.

**💡 면접 포인트**
- "ReAct는 반복적 탐색, planner-executor는 안정적 실행, decompose는 문제 분해에 강하다."
- "Planning 전략은 모델 성능보다 task 특성과 운영 목적에 맞춰 선택해야 한다."
- "Handler registry를 가진 executor는 planner와 tool layer를 느슨하게 연결해 확장성을 높인다."
